# Table A5: AOPC on PhysioMI L/R, finetuned run 3

`script/run_aopc_eval_suite.py` already writes `aopc_table_<run>.csv` and `.tex` (booktabs) next to its results:
mean AOPC, SD across samples, and the ratio to the random baseline.
This notebook regenerates and displays the same table from the results JSON, plus a breakdown by predicted class.

AOPC follows the paper's Eq. 2 (sign from the model's own prediction). Per-sample order: prediction-oriented MoRF;
global order: magnitude of the visualised global heatmap.

In [ ]:
import json, os, sys
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.abspath(".."))
from evaluate.aopc_table import write_aopc_table, METHODS

RESULTS_JSON = "../../out/physiomi_aopc/physiomi_pretrained/aopc_results_global_mean_ps-prediction_gl-magnitude/aopc_evaluation_results.json"
RUN = "run_3"
OUT_PREFIX = "../../out/physiomi_aopc/table_a5"

with open(RESULTS_JSON) as f:
    res = [r for r in json.load(f) if r["run"] == RUN]
assert len(res) == 1, f"Expected exactly one entry for {RUN}, found {len(res)}"
res = res[0]
table = write_aopc_table(res, OUT_PREFIX)
print(f"Wrote {OUT_PREFIX}.csv and {OUT_PREFIX}.tex")
table.round(3)

In [ ]:
print(open(OUT_PREFIX + ".tex").read())

In [ ]:
# Breakdown by predicted class
pred = np.asarray(res["predicted_class"])
pd.DataFrame({label: {f"predicted {c} (n={int((pred == c).sum())})": np.mean(np.asarray(res["per_sample_aopc"][k])[pred == c])
                      for c in (0, 1)} for k, label in METHODS.items()}).T.round(3)